In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss, classification_report

def train_propensity_model(df, target_col, features):
    """
    Trains and calibrates an XGBoost model to predict the propensity of HCPs 
    prescribing the target drug, specifically tailored for Segments B and C.
    """
    # 1. Data Preparation
    X = df[features]
    y = df[target_col]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # 2. Base XGBoost Model
    xgb_base = xgb.XGBClassifier(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42
    )
    
    xgb_base.fit(X_train, y_train)
    
    # 3. Probability Calibration (Crucial for Business/Propensity Scoring)
    # Isotonic regression is generally best if you have enough data
    calibrated_xgb = CalibratedClassifierCV(estimator=xgb_base, method='isotonic', cv='prefit')
    calibrated_xgb.fit(X_test, y_test)
    
    # 4. Evaluation
    propensity_scores = calibrated_xgb.predict_proba(X_test)[:, 1]
    predictions = calibrated_xgb.predict(X_test)
    
    print("--- Model Performance ---")
    print(f"ROC-AUC Score: {roc_auc_score(y_test, propensity_scores):.4f}")
    print(f"Brier Score (Closer to 0 is better for true probabilities): {brier_score_loss(y_test, propensity_scores):.4f}")
    print("\nClassification Report:\n", classification_report(y_test, predictions))
    
    return xgb_base, calibrated_xgb, X_test

def generate_shap_explanations(base_model, X_test, hcp_index=0):
    """
    Generates SHAP values to open the "Glass Box" and explain the predictions
    to the sales and marketing teams.
    """
    # Initialize the Tree Explainer using the base XGBoost model
    explainer = shap.TreeExplainer(base_model)
    shap_values = explainer.shap_values(X_test)
    
    # --- Global Explanation (For Brand Managers / Strategy) ---
    print("\nGenerating Global Feature Importance (Summary Plot)...")
    plt.figure(figsize=(10, 6))
    plt.title("Top Variables Driving Pfizer Prescriptions (Global)")
    shap.summary_plot(shap_values, X_test, show=False)
    plt.tight_layout()
    plt.show()
    
    # --- Local Explanation (For Sales Reps / Individual HCP) ---
    print(f"\nGenerating Individual Explanation for HCP at index {hcp_index}...")
    
    # Setup for Waterfall plot
    explanation = shap.Explanation(
        values=shap_values[hcp_index], 
        base_values=explainer.expected_value, 
        data=X_test.iloc[hcp_index], 
        feature_names=X_test.columns
    )
    
    plt.figure(figsize=(8, 5))
    plt.title(f"Propensity Drivers for HCP #{hcp_index}")
    shap.plots.waterfall(explanation, max_display=10, show=False)
    plt.tight_layout()
    plt.show()

# ==========================================
# Example Usage:
# ==========================================
# Assuming 'hcp_data' is your DataFrame containing only Segments B and C
# target = 'prescribed_pfizer' 
# feature_columns = ['trx_volume', 'digital_details', 'face_to_face_visits', 'specialty_score', ...]

# xgb_base, calibrated_model, X_test = train_propensity_model(hcp_data, target, feature_columns)
# generate_shap_explanations(xgb_base, X_test, hcp_index=42)